# nginx proxy_set_header 설정 완전 가이드

## 목차

1. [HTTP 헤더와 프록시의 기본 개념](#1-http-헤더와-프록시의-기본-개념)
2. [proxy_set_header Upgrade](#2-proxy_set_header-upgrade)
3. [proxy_set_header Host](#3-proxy_set_header-host)
4. [proxy_set_header X-Real-IP](#4-proxy_set_header-x-real-ip)
5. [proxy_set_header X-Forwarded-For](#5-proxy_set_header-x-forwarded-for)
6. [proxy_set_header X-Forwarded-Proto](#6-proxy_set_header-x-forwarded-proto)
7. [통합 예제 및 모범 사례](#7-통합-예제-및-모범-사례)

---



# 1. HTTP 헤더와 프록시의 기본 개념

## 1.1 HTTP 헤더란 무엇인가

HTTP 헤더(HTTP Header)는 HTTP 요청과 응답에 포함되는 메타데이터입니다. 클라이언트와 서버 간의 통신에 필요한 추가 정보를 전달하는 역할을 합니다.

### 주요 특징
- **키-값 쌍(Key-Value Pair) 형식**: `Header-Name: Header-Value`
- **요청 헤더(Request Header)**: 클라이언트가 서버로 보내는 정보
- **응답 헤더(Response Header)**: 서버가 클라이언트로 보내는 정보

### 예시
```
Host: example.com
User-Agent: Mozilla/5.0
Content-Type: application/json
```

## 1.2 프록시의 역할과 동작 원리

프록시(Proxy)는 클라이언트와 서버 사이에서 중간자 역할을 하는 서버입니다.

### 프록시의 주요 기능
1. **요청 전달**: 클라이언트의 요청을 백엔드 서버로 전달
2. **응답 중계**: 백엔드 서버의 응답을 클라이언트로 전달
3. **로드 밸런싱**: 여러 백엔드 서버에 요청 분산
4. **캐싱**: 자주 요청되는 내용을 저장하여 성능 향상
5. **보안**: 클라이언트 IP를 숨기거나 필터링 수행

### 프록시 동작 흐름
```
클라이언트 → 프록시 서버 → 백엔드 서버
         ←             ←
```

## 1.3 nginx 프록시의 기본 동작 방식

nginx는 `proxy_pass` 지시어를 사용하여 프록시 기능을 구현합니다.

### 기본 설정 구조
```nginx
location / {
    proxy_pass http://backend_server;
}
```

### nginx의 기본 헤더 전달 동작
- nginx는 기본적으로 클라이언트가 보낸 **모든 헤더를 그대로** 백엔드로 전달합니다
- 하지만 프록시 환경에서는 **원본 정보가 손실**될 수 있습니다
- 예: 클라이언트의 실제 IP는 프록시의 IP로 대체됨

### proxy_set_header의 필요성
프록시 환경에서 원본 정보를 보존하고 올바른 헤더를 전달하기 위해 `proxy_set_header`를 사용합니다.

---


# 2. proxy_set_header Upgrade

## 2.1 Upgrade 헤더의 의미와 용도

`Upgrade` 헤더는 HTTP 프로토콜에서 다른 프로토콜로 전환하고자 할 때 사용됩니다.

### 주요 용도
- **WebSocket 연결**: HTTP에서 WebSocket 프로토콜로 전환
- **HTTP/2 업그레이드**: HTTP/1.1에서 HTTP/2로 전환
- **기타 프로토콜 전환**: 필요한 경우 다른 프로토콜로 전환

### 헤더 형식
```
Upgrade: websocket
Connection: Upgrade
```

**중요**: `Upgrade` 헤더는 반드시 `Connection: Upgrade`와 함께 사용되어야 합니다.

## 2.2 WebSocket 연결에서의 역할

WebSocket은 실시간 양방향 통신을 위한 프로토콜입니다. HTTP 핸드셰이크를 통해 WebSocket으로 전환됩니다.

### WebSocket 핸드셰이크 과정
1. 클라이언트가 `Upgrade: websocket` 헤더와 함께 요청
2. 서버가 `101 Switching Protocols` 응답
3. HTTP 연결이 WebSocket 연결로 전환

## 2.3 브라우저에서의 동작

### 브라우저가 Upgrade 헤더를 추가하는 경우
브라우저는 **직접적으로** `Upgrade` 헤더를 추가하지 않습니다. 대신:

1. **JavaScript WebSocket API 사용 시**:
   ```javascript
   const ws = new WebSocket('ws://example.com/socket');
   ```
   - 브라우저가 자동으로 `Upgrade: websocket` 헤더 추가
   - `Connection: Upgrade` 헤더도 자동 추가

2. **브라우저의 동작**:
   - WebSocket 연결 시도 시 자동으로 필요한 헤더 생성
   - 개발자가 직접 헤더를 설정할 수 없음 (보안상 이유)

### 실제 요청 예시
```
GET /socket HTTP/1.1
Host: example.com
Upgrade: websocket
Connection: Upgrade
Sec-WebSocket-Key: dGhlIHNhbXBsZSBub25jZQ==
Sec-WebSocket-Version: 13
```

## 2.4 nginx에서의 처리 방식

### 문제 상황
nginx는 기본적으로 `Upgrade`와 `Connection` 헤더를 **제거**할 수 있습니다. 이는 WebSocket 연결이 실패하는 원인이 됩니다.

### 해결 방법
```nginx
proxy_set_header Upgrade $http_upgrade;
proxy_set_header Connection "upgrade";
```

### 설정 설명
- `$http_upgrade`: 클라이언트가 보낸 `Upgrade` 헤더의 값을 변수로 저장
- `proxy_set_header Upgrade $http_upgrade`: 클라이언트의 Upgrade 헤더 값을 그대로 백엔드로 전달
- `proxy_set_header Connection "upgrade"`: Connection 헤더를 명시적으로 설정

### nginx 변수 설명
- `$http_<header_name>`: 클라이언트가 보낸 특정 헤더 값을 읽는 변수
- `$http_upgrade`: `Upgrade` 헤더의 값을 의미
- 값이 없으면 빈 문자열이 됨

### 완전한 WebSocket 프록시 설정
```nginx
location /socket {
    proxy_pass http://backend_server;
    proxy_http_version 1.1;
    proxy_set_header Upgrade $http_upgrade;
    proxy_set_header Connection "upgrade";
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
}
```

## 2.5 백엔드 프록시에서의 처리

### 백엔드가 받는 헤더
nginx가 설정을 올바르게 했다면, 백엔드는 다음과 같은 헤더를 받습니다:

```
Upgrade: websocket
Connection: upgrade
Host: example.com
```

### 백엔드의 동작
1. `Upgrade: websocket` 헤더 확인
2. `Connection: upgrade` 헤더 확인
3. WebSocket 핸드셰이크 수행
4. `101 Switching Protocols` 응답 전송
5. WebSocket 연결로 전환

### 주의사항
- 백엔드가 `Upgrade` 헤더를 받지 못하면 일반 HTTP 요청으로 처리됨
- WebSocket 연결이 실패하고 HTTP 요청으로 처리됨
- 따라서 nginx에서 반드시 헤더를 전달해야 함

---


# 3. proxy_set_header Host

## 3.1 Host 헤더의 의미와 중요성

`Host` 헤더는 HTTP/1.1에서 필수 헤더이며, 요청이 전송되는 대상 서버의 호스트명과 포트를 지정합니다.

### Host 헤더 형식
```
Host: example.com
Host: example.com:8080
```

### 중요성
1. **가상 호스팅(Virtual Hosting)**: 하나의 서버에서 여러 도메인을 구분
2. **라우팅**: 서버가 요청을 어떤 애플리케이션으로 보낼지 결정
3. **SSL/TLS 인증서**: 인증서와 도메인 매칭 확인

## 3.2 가상 호스팅에서의 역할

### 가상 호스팅이란?
하나의 서버에서 여러 도메인을 운영하는 기술입니다.

### 동작 원리
서버는 `Host` 헤더를 확인하여:
- 어떤 도메인으로 요청이 왔는지 판단
- 해당 도메인에 맞는 설정/애플리케이션으로 요청 전달

### 예시
```
요청 1: Host: kotlip.kr        → 애플리케이션 A
요청 2: Host: another.com      → 애플리케이션 B
```

## 3.3 브라우저 요청 시 Host 헤더 동작

### 브라우저의 자동 생성
브라우저는 URL을 기반으로 자동으로 `Host` 헤더를 생성합니다.

### 예시
- URL: `https://kotlip.kr/api/docs`
- 브라우저가 생성하는 헤더: `Host: kotlip.kr`

### 브라우저가 Host 헤더를 추가하는 과정
1. 사용자가 URL을 입력하거나 링크 클릭
2. 브라우저가 URL에서 호스트명 추출
3. HTTP 요청에 `Host: <호스트명>` 헤더 자동 추가
4. 개발자가 직접 설정할 필요 없음

### 실제 요청 예시
```
GET /api/docs HTTP/1.1
Host: kotlip.kr
User-Agent: Mozilla/5.0...
```

## 3.4 nginx에서 Host 헤더 변경의 필요성

### 문제 상황
프록시 환경에서 클라이언트가 보낸 `Host` 헤더를 그대로 전달하면 문제가 발생할 수 있습니다.

### 시나리오
```
클라이언트 요청: Host: kotlip.kr
    ↓
nginx 프록시
    ↓
백엔드 서버: proxy1.aiserv.ktcloud.com
```

### 문제점
- 백엔드 서버는 `Host: proxy1.aiserv.ktcloud.com`을 기대
- 하지만 클라이언트의 `Host: kotlip.kr`이 전달됨
- 백엔드가 Host 헤더로 라우팅/검증을 하면 실패

## 3.5 $host vs 명시적 Host 값

### $host 변수
- nginx 변수로 클라이언트가 보낸 `Host` 헤더 값을 의미
- 예: 클라이언트가 `Host: kotlip.kr`을 보내면 `$host = "kotlip.kr"`

### 명시적 Host 값
- 백엔드가 기대하는 정확한 Host 값을 직접 지정
- 예: `proxy_set_header Host proxy1.aiserv.ktcloud.com;`

### 비교
```nginx
# 방법 1: 클라이언트의 Host를 그대로 전달
proxy_set_header Host $host;  # kotlip.kr이 그대로 전달됨

# 방법 2: 백엔드가 기대하는 Host로 변경
proxy_set_header Host proxy1.aiserv.ktcloud.com;  # 백엔드가 기대하는 값으로 전달
```

## 3.6 백엔드에서 Host 헤더를 사용하는 방식

### 백엔드의 Host 헤더 활용
1. **가상 호스팅 구분**: 여러 도메인을 하나의 서버에서 처리
2. **라우팅**: Host 값에 따라 다른 애플리케이션으로 라우팅
3. **검증**: 허용된 Host만 처리하도록 검증
4. **인증서 매칭**: SSL/TLS 인증서와 도메인 매칭 확인

### 백엔드가 받는 헤더 예시
```
# 올바른 경우
Host: proxy1.aiserv.ktcloud.com
→ 백엔드가 정상적으로 처리

# 잘못된 경우
Host: kotlip.kr
→ 백엔드가 인식하지 못하거나 거부
```

## 3.7 실제 문제 사례 분석

### 문제 상황
사용자가 제공한 실제 문제: `kotlip.kr`에서 `proxy1.aiserv.ktcloud.com`으로 프록시

### 이전 설정 (문제)
```nginx
location /api {
    proxy_pass https://api_backend/api;
    proxy_set_header Host $host;  # kotlip.kr을 그대로 전달
}
```

### 문제점 분석
1. 클라이언트 요청: `Host: kotlip.kr`
2. nginx 전달: `Host: kotlip.kr` (그대로 전달)
3. 백엔드 수신: `Host: kotlip.kr`
4. **백엔드가 `proxy1.aiserv.ktcloud.com`을 기대하므로 실패**

### 현재 설정 (해결)
```nginx
location /api {
    proxy_pass https://api_backend;
    proxy_set_header Host proxy1.aiserv.ktcloud.com;
}
```

### 해결 과정 분석
1. 클라이언트 요청: `Host: kotlip.kr`
2. nginx 처리: Host 헤더를 `proxy1.aiserv.ktcloud.com`으로 변경
3. 백엔드 수신: `Host: proxy1.aiserv.ktcloud.com`
4. **백엔드가 올바른 Host를 받아 정상 처리**

### 추가로 수정된 부분: proxy_pass 경로

#### 이전 설정의 문제
```nginx
proxy_pass https://api_backend/api;
```
- 경로가 중복되거나 잘못 전달될 수 있음
- `/api/docs` 요청이 `/api/api/docs`로 전달될 수 있음

#### 현재 설정
```nginx
proxy_pass https://api_backend;
```
- 전체 경로(`/api/docs`)가 그대로 전달됨
- 백엔드가 올바른 경로를 받음

### 결론
- `Host` 헤더를 백엔드가 기대하는 값으로 설정해야 요청이 정상 처리됩니다
- `$host`를 사용하면 클라이언트의 Host가 그대로 전달되어 문제가 발생할 수 있습니다
- 백엔드의 가상 호스팅 설정에 맞춰 명시적으로 Host를 지정하는 것이 안전합니다

---


# 4. proxy_set_header X-Real-IP

## 4.1 X-Real-IP 헤더의 의미

`X-Real-IP`는 비표준 HTTP 헤더로, 프록시 환경에서 클라이언트의 실제 IP 주소를 전달하기 위해 사용됩니다.

### 헤더 형식
```
X-Real-IP: 192.168.1.100
```

### 용도
- 프록시를 통과한 요청의 실제 클라이언트 IP 주소 보존
- 로깅, 보안, 지역 기반 서비스 등에 활용

## 4.2 실제 클라이언트 IP 추적의 필요성

### 문제 상황
프록시 없이 직접 연결:
```
클라이언트 (192.168.1.100) → 서버
```
서버는 `$remote_addr = 192.168.1.100`을 받음

프록시를 통과:
```
클라이언트 (192.168.1.100) → nginx 프록시 (10.0.0.1) → 백엔드 서버
```
백엔드는 `$remote_addr = 10.0.0.1` (nginx의 IP)만 받음

### 문제점
- 백엔드가 클라이언트의 실제 IP를 알 수 없음
- 로깅에 프록시 IP만 기록됨
- IP 기반 보안 정책 적용 불가
- 지역 기반 서비스 제공 불가

## 4.3 브라우저 요청과 IP 정보

### 브라우저의 동작
브라우저는 **IP 주소 정보를 직접 전달하지 않습니다**. 

### 이유
- IP 주소는 TCP/IP 레벨의 정보
- HTTP 헤더는 애플리케이션 레벨의 정보
- 브라우저는 자신의 IP를 알 수 없음 (서버가 알아야 함)

### 실제 상황
- 클라이언트가 요청을 보낼 때 IP 정보는 포함되지 않음
- 서버가 TCP 연결 정보에서 클라이언트 IP를 확인
- 프록시 환경에서는 이 정보가 손실됨

## 4.4 nginx에서 $remote_addr 변수의 의미

### $remote_addr 변수
- nginx 변수로 **직접 연결된 클라이언트의 IP 주소**를 의미
- TCP 연결에서 얻은 실제 IP 주소

### 동작 방식
```
클라이언트 (192.168.1.100) → nginx
```
nginx에서 `$remote_addr = "192.168.1.100"`

### 프록시 환경에서의 문제
```
클라이언트 → nginx (10.0.0.1) → 백엔드
```
- nginx는 클라이언트 IP를 알 수 있음 (`$remote_addr = 클라이언트 IP`)
- 하지만 백엔드는 nginx의 IP만 받음 (`$remote_addr = 10.0.0.1`)

## 4.5 nginx 설정 방법

### 기본 설정
```nginx
proxy_set_header X-Real-IP $remote_addr;
```

### 동작 원리
1. nginx가 클라이언트로부터 직접 연결을 받음
2. `$remote_addr`에 클라이언트의 실제 IP가 저장됨
3. `X-Real-IP` 헤더에 이 값을 설정하여 백엔드로 전달

### 예시
```nginx
location / {
    proxy_pass http://backend_server;
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
}
```

### 전달 과정
```
클라이언트 (192.168.1.100) 요청
    ↓
nginx: $remote_addr = "192.168.1.100"
    ↓
X-Real-IP: 192.168.1.100 헤더 추가
    ↓
백엔드: X-Real-IP 헤더에서 클라이언트 IP 확인
```

## 4.6 백엔드에서 X-Real-IP 활용 방법

### 백엔드가 받는 헤더
```
X-Real-IP: 192.168.1.100
Host: example.com
```

### 활용 사례

#### 1. 로깅
```python
# Python 예시
client_ip = request.headers.get('X-Real-IP', request.remote_addr)
logger.info(f"Request from {client_ip}")
```

#### 2. 접근 제어
```python
# 허용된 IP만 접근 허용
allowed_ips = ['192.168.1.0/24']
client_ip = request.headers.get('X-Real-IP')
if not is_ip_allowed(client_ip, allowed_ips):
    return "Access Denied", 403
```

#### 3. 지역 기반 서비스
```python
# 클라이언트 IP 기반 지역 판단
client_ip = request.headers.get('X-Real-IP')
region = get_region_from_ip(client_ip)
```

### 주의사항
- `X-Real-IP`는 비표준 헤더이므로 신뢰할 수 있는 프록시에서만 사용
- 클라이언트가 직접 헤더를 조작할 수 있음
- 보안이 중요한 경우 추가 검증 필요

### 다중 프록시 환경
- 첫 번째 프록시에서만 실제 IP를 설정
- 이후 프록시는 값을 그대로 전달
- `X-Forwarded-For`와 함께 사용하는 것이 권장됨

---


# 5. proxy_set_header X-Forwarded-For

## 5.1 X-Forwarded-For 헤더의 의미

`X-Forwarded-For`는 프록시를 통과한 요청의 클라이언트 IP 주소 체인을 기록하는 비표준 HTTP 헤더입니다.

### 헤더 형식
```
X-Forwarded-For: 192.168.1.100
X-Forwarded-For: 192.168.1.100, 10.0.0.1, 203.0.113.5
```

### 용도
- 다중 프록시 환경에서 클라이언트 IP 추적
- 프록시 경로 전체 추적
- IP 체인을 통한 요청 경로 분석

## 5.2 다중 프록시 환경에서의 IP 체인

### 단일 프록시
```
클라이언트 (192.168.1.100) → nginx → 백엔드
```
`X-Forwarded-For: 192.168.1.100`

### 다중 프록시
```
클라이언트 (192.168.1.100) → 프록시1 (10.0.0.1) → 프록시2 (10.0.0.2) → 백엔드
```
각 프록시가 IP를 추가:
- 프록시1: `X-Forwarded-For: 192.168.1.100`
- 프록시2: `X-Forwarded-For: 192.168.1.100, 10.0.0.1`
- 백엔드 수신: `X-Forwarded-For: 192.168.1.100, 10.0.0.1, 10.0.0.2`

### IP 체인의 의미
- 첫 번째 IP: 원본 클라이언트 IP (가장 중요)
- 이후 IP들: 프록시 서버들의 IP (경로 추적용)

## 5.3 $proxy_add_x_forwarded_for 변수의 동작

### 변수 설명
`$proxy_add_x_forwarded_for`는 nginx의 특수 변수로, 다음을 수행합니다:
1. 기존 `X-Forwarded-For` 헤더가 있으면 그 값을 유지
2. 현재 클라이언트의 IP를 콤마로 구분하여 추가
3. 헤더가 없으면 클라이언트 IP만 추가

### 동작 예시

#### 시나리오 1: X-Forwarded-For가 없는 경우
```
클라이언트 (192.168.1.100) 요청 (헤더 없음)
    ↓
nginx: $proxy_add_x_forwarded_for = "192.168.1.100"
    ↓
X-Forwarded-For: 192.168.1.100
```

#### 시나리오 2: X-Forwarded-For가 이미 있는 경우
```
클라이언트 요청: X-Forwarded-For: 203.0.113.5
    ↓
nginx: $proxy_add_x_forwarded_for = "203.0.113.5, 192.168.1.100"
    ↓
X-Forwarded-For: 203.0.113.5, 192.168.1.100
```

### 수동 설정과의 차이
```nginx
# 방법 1: 수동 설정 (문제 발생 가능)
proxy_set_header X-Forwarded-For $remote_addr;
# → 기존 값이 있으면 덮어씀 (손실)

# 방법 2: 자동 추가 (권장)
proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
# → 기존 값을 유지하고 추가
```

## 5.4 브라우저에서의 처리

### 브라우저의 동작
브라우저는 **X-Forwarded-For 헤더를 생성하지 않습니다**.

### 이유
- X-Forwarded-For는 프록시 서버가 설정하는 헤더
- 브라우저는 자신이 프록시인지 알 수 없음
- 클라이언트가 직접 설정하면 신뢰성 문제 발생

### 실제 요청
브라우저가 보내는 요청에는 `X-Forwarded-For` 헤더가 없습니다.

## 5.5 nginx에서의 추가/추적 방식

### 기본 설정
```nginx
proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
```

### 동작 과정

#### 단계별 설명
1. **클라이언트 요청 수신**
   - nginx가 클라이언트로부터 요청을 받음
   - `$remote_addr`에 클라이언트 IP 저장

2. **기존 헤더 확인**
   - 요청에 `X-Forwarded-For` 헤더가 있는지 확인
   - 있으면: 기존 값 + 현재 클라이언트 IP
   - 없으면: 현재 클라이언트 IP만

3. **헤더 설정**
   - `$proxy_add_x_forwarded_for` 변수 사용
   - 자동으로 IP 체인 구성

4. **백엔드로 전달**
   - 구성된 `X-Forwarded-For` 헤더를 백엔드로 전달

### 예시 설정
```nginx
location / {
    proxy_pass http://backend_server;
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
}
```

### 실제 동작 예시

#### 예시 1: 첫 번째 프록시
```
클라이언트 (192.168.1.100) → nginx1
```
nginx1 설정:
- `$remote_addr = "192.168.1.100"`
- `$proxy_add_x_forwarded_for = "192.168.1.100"`
- 전달: `X-Forwarded-For: 192.168.1.100`

#### 예시 2: 두 번째 프록시
```
nginx1 → nginx2
nginx2가 받는 요청: X-Forwarded-For: 192.168.1.100
```
nginx2 설정:
- `$remote_addr = "10.0.0.1"` (nginx1의 IP)
- `$proxy_add_x_forwarded_for = "192.168.1.100, 10.0.0.1"`
- 전달: `X-Forwarded-For: 192.168.1.100, 10.0.0.1`

## 5.6 백엔드에서 IP 체인 파싱 방법

### 백엔드가 받는 헤더
```
X-Forwarded-For: 192.168.1.100, 10.0.0.1, 10.0.0.2
```

### IP 체인 파싱

#### Python 예시
```python
def get_client_ip(request):
    x_forwarded_for = request.headers.get('X-Forwarded-For', '')
    if x_forwarded_for:
        # 첫 번째 IP가 실제 클라이언트 IP
        client_ip = x_forwarded_for.split(',')[0].strip()
    else:
        client_ip = request.remote_addr
    return client_ip
```

#### Node.js 예시
```javascript
function getClientIP(req) {
    const xForwardedFor = req.headers['x-forwarded-for'];
    if (xForwardedFor) {
        // 첫 번째 IP가 실제 클라이언트 IP
        return xForwardedFor.split(',')[0].trim();
    }
    return req.connection.remoteAddress;
}
```

### 주의사항
1. **첫 번째 IP가 실제 클라이언트 IP**: IP 체인에서 가장 왼쪽의 IP
2. **신뢰할 수 있는 프록시만 사용**: 클라이언트가 헤더를 조작할 수 있음
3. **X-Real-IP와 함께 사용**: 더 정확한 IP 추적 가능

### 보안 고려사항
- 클라이언트가 직접 `X-Forwarded-For`를 설정할 수 있음
- 신뢰할 수 있는 프록시에서만 이 헤더를 설정해야 함
- 보안이 중요한 경우 IP 화이트리스트 사용 권장

### X-Real-IP와의 비교
- **X-Real-IP**: 단일 IP (프록시 하나만 있을 때 사용)
- **X-Forwarded-For**: IP 체인 (다중 프록시 환경에서 사용)
- **권장**: 두 헤더를 모두 사용하여 유연성 확보

---


# 6. proxy_set_header X-Forwarded-Proto

## 6.1 X-Forwarded-Proto 헤더의 의미

`X-Forwarded-Proto`는 프록시를 통과한 요청의 원본 프로토콜(HTTP 또는 HTTPS)을 전달하는 비표준 HTTP 헤더입니다.

### 헤더 형식
```
X-Forwarded-Proto: https
X-Forwarded-Proto: http
```

### 용도
- SSL/TLS 종료(SSL Termination) 환경에서 원본 프로토콜 정보 보존
- 백엔드가 HTTPS인지 HTTP인지 판단
- 리다이렉트 URL 생성 시 올바른 프로토콜 사용

## 6.2 HTTPS 종료(SSL Termination) 환경

### SSL Termination이란?
프록시 서버에서 SSL/TLS를 종료하고, 백엔드 서버와는 일반 HTTP로 통신하는 방식입니다.

### 아키텍처
```
클라이언트 → [HTTPS] → nginx 프록시 → [HTTP] → 백엔드 서버
                    (SSL 종료)
```

### 장점
1. **성능 향상**: SSL 암호화/복호화 부하를 프록시에서만 처리
2. **인증서 관리**: 프록시에서만 인증서 관리
3. **백엔드 단순화**: 백엔드는 HTTP만 처리

### 문제점
- 백엔드는 HTTP로 요청을 받지만, 실제로는 HTTPS 요청이었음
- 백엔드가 프로토콜을 구분할 수 없음

## 6.3 $scheme 변수의 의미

### $scheme 변수
nginx 변수로, **요청이 들어온 프로토콜**을 의미합니다.

### 값
- `$scheme = "http"`: HTTP 요청
- `$scheme = "https"`: HTTPS 요청

### 동작 방식
```
클라이언트 → [HTTPS] → nginx
```
nginx에서 `$scheme = "https"`

```
클라이언트 → [HTTP] → nginx
```
nginx에서 `$scheme = "http"`

## 6.4 브라우저 요청과 프로토콜 정보

### 브라우저의 동작
브라우저는 URL의 프로토콜을 확인하여 요청을 보냅니다.

### 예시
- URL: `https://example.com/api`
- 브라우저는 HTTPS 연결을 시도
- HTTP 요청에는 프로토콜 정보가 헤더에 포함되지 않음

### 실제 요청
```
GET /api HTTP/1.1
Host: example.com
```
프로토콜 정보는 TCP 레벨에서만 확인 가능

### 문제
프록시를 통과하면:
- 백엔드는 HTTP로 요청을 받음
- 원본이 HTTPS였는지 알 수 없음

## 6.5 nginx에서의 처리

### 기본 설정
```nginx
proxy_set_header X-Forwarded-Proto $scheme;
```

### 동작 원리
1. nginx가 클라이언트로부터 요청을 받음
2. `$scheme`에 프로토콜 정보 저장 (http 또는 https)
3. `X-Forwarded-Proto` 헤더에 이 값을 설정하여 백엔드로 전달

### 예시 설정
```nginx
location / {
    proxy_pass http://backend_server;
    proxy_set_header Host $host;
    proxy_set_header X-Forwarded-Proto $scheme;
}
```

### 전달 과정

#### HTTPS 요청의 경우
```
클라이언트 → [HTTPS] → nginx
    ↓
nginx: $scheme = "https"
    ↓
X-Forwarded-Proto: https 헤더 추가
    ↓
백엔드: HTTP로 요청을 받지만, X-Forwarded-Proto로 원본 프로토콜 확인
```

#### HTTP 요청의 경우
```
클라이언트 → [HTTP] → nginx
    ↓
nginx: $scheme = "http"
    ↓
X-Forwarded-Proto: http 헤더 추가
    ↓
백엔드: HTTP 요청임을 확인
```

### SSL Termination 설정 예시
```nginx
server {
    listen 443 ssl;
    server_name example.com;
    
    ssl_certificate /path/to/cert.pem;
    ssl_certificate_key /path/to/key.pem;
    
    location / {
        proxy_pass http://backend_server;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;  # https로 설정됨
    }
}
```

## 6.6 백엔드에서의 프로토콜 인식 방법

### 백엔드가 받는 헤더
```
X-Forwarded-Proto: https
Host: example.com
```

### 활용 사례

#### 1. 리다이렉트 URL 생성
```python
# Python 예시
def get_redirect_url(path):
    proto = request.headers.get('X-Forwarded-Proto', 'http')
    host = request.headers.get('Host', 'localhost')
    return f"{proto}://{host}{path}"

# 예: https://example.com/login으로 리다이렉트
```

#### 2. 보안 검증
```python
# HTTPS만 허용
proto = request.headers.get('X-Forwarded-Proto', 'http')
if proto != 'https':
    return "HTTPS required", 403
```

#### 3. 쿠키 설정
```python
# Secure 쿠키 설정 (HTTPS인 경우)
proto = request.headers.get('X-Forwarded-Proto', 'http')
secure = (proto == 'https')
response.set_cookie('session', value, secure=secure)
```

#### 4. 로깅
```python
# 프로토콜 정보 포함 로깅
proto = request.headers.get('X-Forwarded-Proto', 'http')
logger.info(f"{proto.upper()} request from {client_ip}")
```

### Node.js 예시
```javascript
// Express.js 예시
app.use((req, res, next) => {
    const proto = req.headers['x-forwarded-proto'] || 'http';
    req.protocol = proto;  // req.protocol 사용 가능
    next();
});

// 리다이렉트
app.get('/login', (req, res) => {
    const url = `${req.protocol}://${req.get('host')}/auth`;
    res.redirect(url);
});
```

### 주의사항
1. **신뢰할 수 있는 프록시**: 클라이언트가 헤더를 조작할 수 있음
2. **기본값 처리**: 헤더가 없을 경우 HTTP로 가정
3. **보안**: 중요한 경우 추가 검증 필요

### 다중 프록시 환경
- 첫 번째 프록시에서 원본 프로토콜 설정
- 이후 프록시는 값을 그대로 전달
- 각 프록시의 프로토콜이 아닌 원본 프로토콜이 중요

### 실제 사용 예시
```nginx
# HTTP와 HTTPS 모두 처리
server {
    listen 80;
    listen 443 ssl;
    
    location / {
        proxy_pass http://backend;
        proxy_set_header X-Forwarded-Proto $scheme;
        # HTTP 요청: X-Forwarded-Proto: http
        # HTTPS 요청: X-Forwarded-Proto: https
    }
}
```

---


# 7. 통합 예제 및 모범 사례

## 7.1 실제 nginx 설정 예제

### 기본 프록시 설정
```nginx
location / {
    proxy_pass http://backend_server;
    
    # 기본 헤더 설정
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    proxy_set_header X-Forwarded-Proto $scheme;
    
    # 타임아웃 설정
    proxy_connect_timeout 60s;
    proxy_send_timeout 60s;
    proxy_read_timeout 60s;
}
```

### WebSocket 지원 프록시 설정
```nginx
location /socket {
    proxy_pass http://backend_server;
    
    # WebSocket을 위한 HTTP 버전 업그레이드
    proxy_http_version 1.1;
    proxy_set_header Upgrade $http_upgrade;
    proxy_set_header Connection "upgrade";
    
    # 기본 헤더 설정
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    proxy_set_header X-Forwarded-Proto $scheme;
    
    # WebSocket 타임아웃 (연결 유지)
    proxy_read_timeout 3600s;
}
```

### SSL Termination 환경 설정
```nginx
server {
    listen 443 ssl;
    server_name example.com;
    
    ssl_certificate /path/to/cert.pem;
    ssl_certificate_key /path/to/key.pem;
    
    location / {
        proxy_pass http://backend_server;
        
        # Host를 백엔드가 기대하는 값으로 변경
        proxy_set_header Host proxy1.aiserv.ktcloud.com;
        
        # 클라이언트 정보 전달
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;  # https로 설정됨
        
        # 버퍼 설정
        proxy_buffering on;
        proxy_buffer_size 4k;
        proxy_buffers 8 4k;
    }
}
```

## 7.2 모든 헤더를 함께 사용하는 경우

### 완전한 프록시 설정
```nginx
location /api {
    proxy_pass https://backend_server;
    
    # HTTP 버전 (WebSocket 지원)
    proxy_http_version 1.1;
    
    # WebSocket 헤더
    proxy_set_header Upgrade $http_upgrade;
    proxy_set_header Connection $connection_upgrade;
    
    # Host 헤더 (백엔드가 기대하는 값으로 설정)
    proxy_set_header Host proxy1.aiserv.ktcloud.com;
    
    # 클라이언트 IP 정보
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    
    # 프로토콜 정보
    proxy_set_header X-Forwarded-Proto $scheme;
    
    # 추가 헤더 (선택사항)
    proxy_set_header X-Forwarded-Host $host;
    proxy_set_header X-Forwarded-Port $server_port;
}
```

### Connection 헤더 변수 설정
WebSocket을 사용할 때는 `$connection_upgrade` 변수를 사용하는 것이 좋습니다:
```nginx
map $http_upgrade $connection_upgrade {
    default upgrade;
    '' close;
}

server {
    location / {
        proxy_set_header Connection $connection_upgrade;
    }
}
```

## 7.3 문제 해결 시나리오

### 시나리오 1: Host 헤더 문제

#### 증상
- 백엔드에서 404 또는 라우팅 실패
- 로그에 잘못된 Host 값 확인

#### 원인
```nginx
proxy_set_header Host $host;  # 클라이언트의 Host를 그대로 전달
```

#### 해결
```nginx
proxy_set_header Host proxy1.aiserv.ktcloud.com;  # 백엔드가 기대하는 Host로 설정
```

### 시나리오 2: WebSocket 연결 실패

#### 증상
- WebSocket 연결이 즉시 끊김
- HTTP 요청으로 처리됨

#### 원인
```nginx
# Upgrade 헤더가 전달되지 않음
location /socket {
    proxy_pass http://backend;
}
```

#### 해결
```nginx
location /socket {
    proxy_pass http://backend;
    proxy_http_version 1.1;
    proxy_set_header Upgrade $http_upgrade;
    proxy_set_header Connection "upgrade";
}
```

### 시나리오 3: 클라이언트 IP가 프록시 IP로 표시됨

#### 증상
- 로그에 프록시 IP만 기록됨
- 실제 클라이언트 IP를 알 수 없음

#### 원인
```nginx
# X-Real-IP 또는 X-Forwarded-For 헤더가 설정되지 않음
location / {
    proxy_pass http://backend;
}
```

#### 해결
```nginx
location / {
    proxy_pass http://backend;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
}
```

### 시나리오 4: HTTPS 리다이렉트가 HTTP로 생성됨

#### 증상
- HTTPS로 접속했는데 리다이렉트 URL이 HTTP로 생성됨
- 보안 경고 발생

#### 원인
```nginx
# X-Forwarded-Proto 헤더가 설정되지 않음
location / {
    proxy_pass http://backend;
}
```

#### 해결
```nginx
location / {
    proxy_pass http://backend;
    proxy_set_header X-Forwarded-Proto $scheme;
}
```

## 7.4 주의사항 및 베스트 프랙티스

### 1. Host 헤더 설정
- ✅ **권장**: 백엔드가 기대하는 Host 값을 명시적으로 설정
- ❌ **비권장**: `$host`를 무조건 사용 (백엔드 설정과 맞지 않을 수 있음)

### 2. 클라이언트 IP 전달
- ✅ **권장**: `X-Real-IP`와 `X-Forwarded-For`를 함께 사용
- ✅ **권장**: `$proxy_add_x_forwarded_for` 사용 (기존 값 보존)
- ❌ **비권장**: `X-Forwarded-For`를 수동으로 설정 (기존 값 손실)

### 3. 프로토콜 정보
- ✅ **권장**: SSL Termination 환경에서 반드시 `X-Forwarded-Proto` 설정
- ✅ **권장**: `$scheme` 변수 사용

### 4. WebSocket 지원
- ✅ **권장**: WebSocket 경로에 `Upgrade`와 `Connection` 헤더 설정
- ✅ **권장**: `proxy_http_version 1.1` 사용
- ✅ **권장**: WebSocket 경로에 더 긴 타임아웃 설정

### 5. 보안 고려사항
- ⚠️ **주의**: 클라이언트가 헤더를 조작할 수 있음
- ✅ **권장**: 신뢰할 수 있는 프록시에서만 헤더 설정
- ✅ **권장**: IP 화이트리스트 사용 고려
- ✅ **권장**: 백엔드에서 헤더 검증 로직 구현

### 6. 성능 최적화
- ✅ **권장**: 적절한 버퍼 크기 설정
- ✅ **권장**: 타임아웃 값 조정
- ✅ **권장**: 불필요한 헤더 제거

### 7. 디버깅 팁
- 로그에 받은 헤더 값 기록
- `curl -H` 옵션으로 헤더 직접 테스트
- 백엔드에서 받은 헤더 값 확인
- nginx 변수 값을 로그로 출력

### 8. 설정 검증
```nginx
# 디버깅용 로그 설정
log_format detailed '$remote_addr - $remote_user [$time_local] '
                    '"$request" $status $body_bytes_sent '
                    '"$http_host" "$http_upgrade" '
                    '"$http_x_forwarded_for" "$http_x_forwarded_proto"';

access_log /var/log/nginx/detailed.log detailed;
```

## 7.5 요약

### 핵심 포인트
1. **Host**: 백엔드가 기대하는 값으로 명시적으로 설정
2. **X-Real-IP**: 단일 프록시에서 클라이언트 IP 전달
3. **X-Forwarded-For**: 다중 프록시에서 IP 체인 추적
4. **X-Forwarded-Proto**: SSL Termination 환경에서 프로토콜 정보 전달
5. **Upgrade**: WebSocket 연결 지원

### 설정 체크리스트
- [ ] Host 헤더가 백엔드가 기대하는 값으로 설정되었는가?
- [ ] 클라이언트 IP 정보가 전달되는가? (X-Real-IP 또는 X-Forwarded-For)
- [ ] SSL Termination 환경에서 X-Forwarded-Proto가 설정되었는가?
- [ ] WebSocket이 필요한 경우 Upgrade 헤더가 설정되었는가?
- [ ] 적절한 타임아웃 값이 설정되었는가?
- [ ] 보안 고려사항이 반영되었는가?

---

## 참고 자료

- [nginx proxy_module 문서](http://nginx.org/en/docs/http/ngx_http_proxy_module.html)
- [HTTP 헤더 스펙](https://developer.mozilla.org/ko/docs/Web/HTTP/Headers)
- [WebSocket 프로토콜](https://tools.ietf.org/html/rfc6455)

---

**작성 완료**: 이 가이드는 nginx의 proxy_set_header 설정에 대한 완전한 이해를 제공합니다.
